<a href="https://colab.research.google.com/github/kej534923-maker/ECON5200-Applied-Data-Analytics/blob/main/fomc_sentiment_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [4]:
import nltk

for pkg in ['punkt', 'punkt_tab', 'stopwords', 'wordnet', 'omw-1.4']:
    nltk.download(pkg)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


In [5]:
"""
fomc_sentiment.py — FOMC Text Analysis Module

Reusable functions for preprocessing, sentiment scoring, and
TF-IDF vectorization of Federal Reserve meeting minutes.

Author: Jia Ke
Course: ECON 5200, Lab 23
"""

import re
from typing import Any, Dict, List, Tuple

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer


# Simplified Loughran-McDonald word lists
LM_NEGATIVE = {
    'adverse', 'adversely', 'against', 'concern', 'concerned', 'concerns',
    'decline', 'declined', 'declining', 'decrease', 'decreased', 'deficit',
    'deteriorate', 'deteriorated', 'deteriorating', 'difficult', 'difficulty',
    'downturn', 'fail', 'failure', 'falling', 'loss', 'losses', 'negative',
    'negatively', 'recession', 'recessionary', 'risk', 'risks', 'risky',
    'severe', 'severely', 'slowdown', 'sluggish', 'stress', 'stressed',
    'threat', 'threaten', 'troubled', 'uncertain', 'uncertainty',
    'unfavorable', 'volatile', 'volatility', 'vulnerability', 'vulnerable',
    'weak', 'weaken', 'weakened', 'weakness', 'worse', 'worsen', 'worsened'
}

LM_POSITIVE = {
    'achieve', 'achieved', 'achievement', 'benefit', 'beneficial', 'confidence',
    'confident', 'favorable', 'gain', 'gained', 'gains', 'good', 'growth',
    'improve', 'improved', 'improvement', 'improving', 'increase', 'increased',
    'opportunity', 'optimism', 'optimistic', 'positive', 'positively',
    'profit', 'profitable', 'progress', 'rebound', 'recover', 'recovery',
    'strength', 'strengthen', 'strong', 'stronger', 'success', 'successful'
}

LM_UNCERTAINTY = {
    'approximate', 'approximately', 'assume', 'assumption', 'believe',
    'cautious', 'could', 'depend', 'depends', 'doubt', 'estimate',
    'expect', 'expected', 'forecast', 'indefinite', 'likelihood', 'may',
    'might', 'nearly', 'perhaps', 'possible', 'possibly', 'predict',
    'preliminary', 'probable', 'probably', 'risk', 'roughly', 'seem',
    'suggest', 'tentative', 'uncertain', 'uncertainty', 'unclear',
    'unpredictable', 'variable'
}


_STOP_WORDS = set(stopwords.words('english'))
_LEMMATIZER = WordNetLemmatizer()


def preprocess_fomc(text: str) -> str:
    """Clean and tokenize FOMC minutes text.

    Steps: lowercase, regex clean, word_tokenize, stop words, lemmatize.
    Returns space-joined clean tokens.
    """
    if text is None:
        return ""

    if not isinstance(text, str):
        text = str(text)

    text = text.lower()
    tokens = word_tokenize(text)

    cleaned_tokens = []
    for token in tokens:
        token = re.sub(r'[^a-z]', '', token)
        if token and token not in _STOP_WORDS and len(token) > 2:
            token = _LEMMATIZER.lemmatize(token)
            cleaned_tokens.append(token)

    return " ".join(cleaned_tokens)


def compute_lm_sentiment(text: str) -> Dict[str, Any]:
    """Compute Loughran-McDonald sentiment scores.

    Returns dict with 'net_sentiment', 'uncertainty',
    'neg_count', 'pos_count', 'unc_count', 'total_words'.
    """
    if text is None:
        text = ""

    if not isinstance(text, str):
        text = str(text)

    tokens = text.split()
    total_words = len(tokens)

    if total_words == 0:
        return {
            'net_sentiment': 0.0,
            'uncertainty': 0.0,
            'neg_count': 0,
            'pos_count': 0,
            'unc_count': 0,
            'total_words': 0
        }

    neg_count = sum(1 for token in tokens if token in LM_NEGATIVE)
    pos_count = sum(1 for token in tokens if token in LM_POSITIVE)
    unc_count = sum(1 for token in tokens if token in LM_UNCERTAINTY)

    return {
        'net_sentiment': (pos_count - neg_count) / total_words,
        'uncertainty': unc_count / total_words,
        'neg_count': neg_count,
        'pos_count': pos_count,
        'unc_count': unc_count,
        'total_words': total_words
    }


def build_tfidf_matrix(
    texts: List[str],
    min_df: int = 5,
    max_df: float = 0.85,
    max_features: int = 5000
) -> Tuple:
    """Build TF-IDF matrix from preprocessed texts.

    Returns (sparse_matrix, feature_names, vectorizer).
    """
    if texts is None or len(texts) == 0:
        raise ValueError("texts must be a non-empty list of strings.")

    processed_texts = []
    for text in texts:
        if text is None:
            processed_texts.append("")
        elif isinstance(text, str):
            processed_texts.append(text)
        else:
            processed_texts.append(str(text))

    vectorizer = TfidfVectorizer(
        min_df=min_df,
        max_df=max_df,
        max_features=max_features,
        ngram_range=(1, 2)
    )

    sparse_matrix = vectorizer.fit_transform(processed_texts)
    feature_names = vectorizer.get_feature_names_out()

    return sparse_matrix, feature_names, vectorizer


if __name__ == "__main__":
    test_text = "The committee noted that inflation remained elevated above target."
    clean = preprocess_fomc(test_text)
    print("Preprocessed:", clean)

    sentiment = compute_lm_sentiment(clean)
    print("Sentiment:", sentiment)

    docs = [
        preprocess_fomc("Inflation remained elevated and risks persisted."),
        preprocess_fomc("Economic growth improved and confidence strengthened."),
        preprocess_fomc("The committee expressed uncertainty about future conditions.")
    ]

    X, features, vec = build_tfidf_matrix(docs, min_df=1, max_df=1.0, max_features=20)
    print("TF-IDF shape:", X.shape)
    print("Features:", features[:10])
    print("fomc_sentiment.py loaded successfully.")

Preprocessed: committee noted inflation remained elevated target
Sentiment: {'net_sentiment': 0.0, 'uncertainty': 0.0, 'neg_count': 0, 'pos_count': 0, 'unc_count': 0, 'total_words': 6}
TF-IDF shape: (3, 20)
Features: ['committee' 'committee expressed' 'condition' 'confidence'
 'confidence strengthened' 'economic' 'economic growth' 'elevated'
 'elevated risk' 'expressed']
fomc_sentiment.py loaded successfully.
